# Scraping de Mercadona - Versión Optimizada

Este notebook realiza scraping de la web de Mercadona con procesamiento en paralelo para mayor velocidad.

In [1]:
# Import libraries
# Selenium: Automate web browser interactions.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import requests

# Polars: A high-performance DataFrame library.
import polars as pl
import pandas as pd

# Paralelismo y manejo de tiempo
import time
import concurrent.futures
from tqdm.notebook import tqdm  # Barras de progreso optimizadas para notebooks
import logging

# Sistema operativo
import os
import sys

# Importar funciones originales (por compatibilidad)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.main import *

# Configurar logging para tener mejores mensajes de error
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Funciones Optimizadas para Scraping en Paralelo

In [1]:
import os
import time
import polars as pl
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
from tqdm.notebook import tqdm
import logging
import math

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_optimized_driver():
    """Crear un driver de Chrome optimizado para scraping."""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-extensions")
    
    # Reducir registro innecesario
    options.add_argument("--log-level=3")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.set_page_load_timeout(30)
    return driver

def scrape_single_page(page_num, get_secondary_images=True):
    """Procesa una sola página de Mercadona."""
    driver = get_optimized_driver()
    
    products_data = []
    url = f"https://tienda.mercadona.es/categories/{page_num}"
    
    try:
        # Registrar qué worker está procesando qué página
        logger.info(f"Worker procesando página {page_num}")
        
        # Abrir la URL
        driver.get(url)
        wait = WebDriverWait(driver, 10)
        
        # Manejar código postal si necesario
        try:
            postal_code_input = wait.until(
                EC.presence_of_element_located((By.CLASS_NAME, "ym-hide-content"))
            )
            postal_code_input.send_keys("28039")
            
            submit_button = driver.find_element(By.XPATH, "/html/body/div[1]/div[5]/div/div[2]/div/form/button")
            submit_button.click()
            
            # Espera explícita
            wait.until(EC.invisibility_of_element_located((By.CLASS_NAME, "ym-hide-content")))
        except Exception as e:
            # Ignora errores aquí, probablemente ya esté configurado el código postal
            pass
        
        # Obtener categoría
        try:
            category = wait.until(
                EC.presence_of_element_located((By.CLASS_NAME, "category-detail__title.title1-b"))
            ).text
        except Exception as e:
            logger.warning(f"No se pudo obtener categoría en página {page_num}: {e}")
            category = f"Unknown-{page_num}"
        
        # Obtener productos
        try:
            product_elements = wait.until(
                EC.presence_of_all_elements_located((By.CLASS_NAME, "product-cell__content-link"))
            )
            logger.info(f"Encontrados {len(product_elements)} productos en página {page_num}")
        except Exception as e:
            logger.error(f"No se encontraron productos en página {page_num}: {e}")
            driver.quit()
            return []
        
        # Extraer datos de cada producto
        for product in product_elements:
            product_data = {'Category': category}
            
            # Extraer nombre
            try:
                product_data['name'] = product.find_element(By.CLASS_NAME, "subhead1-r.product-cell__description-name").text
            except Exception as e:
                product_data['name'] = None
            
            # Extraer subtítulo
            try:
                product_data['subtitle'] = product.find_element(By.CLASS_NAME, "product-format.product-format__size--cell").text
            except Exception as e:
                product_data['subtitle'] = None
            
            # Extraer precio
            try:
                product_data['price'] = product.find_element(By.CLASS_NAME, "product-price__unit-price.subhead1-b").text
            except Exception as e:
                product_data['price'] = None
            
            # Extraer precio con descuento
            try:
                product_data['discount_price'] = product.find_element(By.CLASS_NAME, "product-price__unit-price--discount").text
            except Exception as e:
                product_data['discount_price'] = None
            
            # Extraer imagen principal
            try:
                image_element = product.find_element(By.CLASS_NAME, "product-cell__image-wrapper")
                img_element = image_element.find_element(By.TAG_NAME, "img")
                product_data['main_image_url'] = img_element.get_attribute("src")
            except Exception as e:
                product_data['main_image_url'] = None
            
            # Extraer imagen secundaria
            if get_secondary_images:
                try:
                    # Abrir modal del producto
                    product.click()
                    
                    # Esperar a miniaturas
                    image_elements = wait.until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "product-gallery__thumbnail"))
                    )
                    
                    if len(image_elements) > 1:
                        # Hacer clic en la segunda miniatura
                        second_image = image_elements[1]
                        second_image.click()
                        
                        # Esperar a que se cargue la imagen
                        image_zoomer_element = wait.until(
                            EC.presence_of_element_located((By.CLASS_NAME, "image-zoomer__source"))
                        )
                        img_element = image_zoomer_element.find_element(By.TAG_NAME, "img")
                        product_data['secondary_image_url'] = img_element.get_attribute("src")
                    else:
                        product_data['secondary_image_url'] = None
                    
                    # Cerrar modal
                    close_button = driver.find_element(By.CLASS_NAME, "modal-content__close")
                    close_button.click()
                    wait.until(EC.invisibility_of_element_located((By.CLASS_NAME, "modal-content__close")))
                    
                except Exception as e:
                    product_data['secondary_image_url'] = None
            else:
                product_data['secondary_image_url'] = None
            
            products_data.append(product_data)
            
    except Exception as e:
        logger.error(f"Error procesando página {page_num}: {e}")
    finally:
        driver.quit()
    
    return products_data

def worker_process_pages(page_range, get_secondary_images=True):
    """
    Función para que cada worker procese su propio rango de páginas.
    
    Parameters:
    -----------
    page_range : list
        Lista de números de página que este worker debe procesar
    get_secondary_images : bool
        Si se deben obtener imágenes secundarias
        
    Returns:
    --------
    list: Lista de productos encontrados en todas las páginas procesadas
    """
    worker_products = []
    
    for page_num in page_range:
        try:
            page_products = scrape_single_page(page_num, get_secondary_images)
            if page_products:
                worker_products.extend(page_products)
                logger.info(f"Página {page_num}: {len(page_products)} productos encontrados")
        except Exception as e:
            logger.error(f"Error en página {page_num}: {e}")
    
    return worker_products

def scrape_mercadona_distributed(start_page=0, end_page=300, num_workers=5, get_secondary_images=True):
    """
    Versión distribuida del scraping que asigna conjuntos distintos de páginas a cada worker.
    
    Parameters:
    -----------
    start_page : int
        Página inicial
    end_page : int 
        Página final
    num_workers : int
        Número de workers paralelos
    get_secondary_images : bool
        Si se deben obtener imágenes secundarias
    """
    all_products = []
    
    print(f"Iniciando scraping distribuido desde página {start_page} hasta {end_page} con {num_workers} workers")
    
    # Calcular páginas totales y dividirlas entre workers
    total_pages = end_page - start_page + 1
    pages_per_worker = math.ceil(total_pages / num_workers)
    
    # Crear ranges de páginas para cada worker (distribución equitativa)
    page_distributions = []
    for i in range(num_workers):
        worker_start = start_page + (i * pages_per_worker)
        worker_end = min(worker_start + pages_per_worker - 1, end_page)
        
        # Solo añadir si hay páginas válidas en este rango
        if worker_start <= worker_end:
            worker_pages = list(range(worker_start, worker_end + 1))
            page_distributions.append(worker_pages)
            logger.info(f"Worker {i+1} procesará páginas {worker_start} a {worker_end} ({len(worker_pages)} páginas)")
    
    # Verificar que todas las páginas están asignadas
    all_assigned_pages = [p for worker_pages in page_distributions for p in worker_pages]
    logger.info(f"Total de páginas asignadas: {len(all_assigned_pages)}")
    
    # Usar ThreadPoolExecutor para procesamiento paralelo
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        with tqdm(total=total_pages, desc="Scraping Mercadona") as pbar:
            # Enviar cada distribución de páginas a un worker
            futures = []
            for worker_pages in page_distributions:
                future = executor.submit(
                    worker_process_pages, 
                    worker_pages, 
                    get_secondary_images
                )
                futures.append(future)
            
            # Procesar resultados a medida que se completan
            pages_processed = 0
            for future in concurrent.futures.as_completed(futures):
                try:
                    worker_products = future.result()
                    if worker_products:
                        all_products.extend(worker_products)
                        worker_pages_count = len(worker_products)
                        logger.info(f"Worker completado: {worker_pages_count} productos extraídos")
                    
                    # Actualizar la barra de progreso (estimación)
                    pages_per_future = total_pages // len(futures)
                    pbar.update(pages_per_future)
                    pages_processed += pages_per_future
                    
                except Exception as e:
                    logger.error(f"Error en worker: {e}")
            
            # Asegurar que la barra de progreso se completa
            if pages_processed < total_pages:
                pbar.update(total_pages - pages_processed)
    
    # Convertir a DataFrame de Polars
    if all_products:
        print(f"Scraping completado. Total productos: {len(all_products)}")
        df = pl.DataFrame(all_products)
        # Guardar CSV con los datos crudos
        df.write_csv("../data/raw/products.csv")
        return df
    else:
        print("No se encontraron productos")
        return pl.DataFrame()

def download_image_parallel(args):
    """Función para descargar imagen que será llamada en paralelo."""
    url, path = args
    if url:
        try:
            # Asegurar que el directorio existe
            os.makedirs(os.path.dirname(path), exist_ok=True)
            
            # Descargar imagen con timeout
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with open(path, 'wb') as f:
                    f.write(response.content)
                return True
            else:
                logger.warning(f"Error {response.status_code} al descargar {url}")
                return False
        except Exception as e:
            logger.error(f"Error descargando {url}: {str(e)[:100]}")
            return False
    return False

def download_images_parallel(df, base_path="../img", max_workers=10):
    """Descarga todas las imágenes en paralelo."""
    # Crear carpetas de categorías primero
    create_category_folders(df, base_path)
    
    # Preparar tareas de descarga
    download_tasks = []
    
    # Para cada fila, preparar las descargas principales y secundarias
    for row in df.to_pandas().itertuples():
        category = row.Category
        img_id = row.id
        
        # Imagen principal
        if hasattr(row, 'main_image_url') and row.main_image_url:
            main_path = f"{base_path}/{category}/{img_id}.jpg"
            download_tasks.append((row.main_image_url, main_path))
        
        # Imagen secundaria
        if hasattr(row, 'secondary_image_url') and row.secondary_image_url:
            secondary_path = f"{base_path}/{category}/{img_id}_secondary.jpg"
            download_tasks.append((row.secondary_image_url, secondary_path))
    
    print(f"Preparadas {len(download_tasks)} imágenes para descargar")
    
    # Usar ThreadPoolExecutor para descargar en paralelo
    successful_downloads = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Mostrar barra de progreso
        with tqdm(total=len(download_tasks), desc="Descargando imágenes") as pbar:
            for success in executor.map(download_image_parallel, download_tasks):
                if success:
                    successful_downloads += 1
                pbar.update(1)
    
    print(f"Descarga completada. {successful_downloads} de {len(download_tasks)} imágenes descargadas.")

def create_category_folders(df, base_path="../img"):
    """
    Crea una carpeta para cada categoría única en la columna 'Category' del DataFrame.
    """
    try:
        # Asegurar que la carpeta base existe
        os.makedirs(base_path, exist_ok=True)
        
        # Crear carpetas de categorías
        categories = df["Category"].unique().to_list()
        for category in categories:
            category_path = f"{base_path}/{category}"
            os.makedirs(category_path, exist_ok=True)
            
        logger.info(f"Creadas {len(categories)} carpetas de categorías")
    except Exception as e:
        logger.error(f"Error creando carpetas de categorías: {e}")

# Ejemplo de uso para notebook
if __name__ == "__main__":
    # Ejecutar scraping distribuido
    start_time = time.time()
    
    df = scrape_mercadona_distributed(
        start_page=0,
        end_page=1000,
        num_workers=20,
        get_secondary_images=True  # Cambiar a True si necesitas las imágenes secundarias
    )
    
    end_time = time.time()
    print(f"Tiempo total: {end_time - start_time:.2f} segundos")
    
    if not df.is_empty():
        # Agregar IDs
        df = df.with_columns(
            pl.arange(1, df.height + 1).alias("id")
        )
        
        # Reorganizar las columnas
        df = df.select(["id", "Category", "name", "subtitle", "price", "discount_price", "main_image_url", "secondary_image_url"])
        
        # Guardar DataFrame procesado
        df.write_csv("../data/raw/mercadona.csv", index=False)

2025-03-31 16:30:27,092 - INFO - Worker 1 procesará páginas 0 a 50 (51 páginas)
2025-03-31 16:30:27,092 - INFO - Worker 2 procesará páginas 51 a 101 (51 páginas)
2025-03-31 16:30:27,092 - INFO - Worker 3 procesará páginas 102 a 152 (51 páginas)
2025-03-31 16:30:27,092 - INFO - Worker 4 procesará páginas 153 a 203 (51 páginas)
2025-03-31 16:30:27,093 - INFO - Worker 5 procesará páginas 204 a 254 (51 páginas)
2025-03-31 16:30:27,093 - INFO - Worker 6 procesará páginas 255 a 305 (51 páginas)
2025-03-31 16:30:27,093 - INFO - Worker 7 procesará páginas 306 a 356 (51 páginas)
2025-03-31 16:30:27,093 - INFO - Worker 8 procesará páginas 357 a 407 (51 páginas)
2025-03-31 16:30:27,094 - INFO - Worker 9 procesará páginas 408 a 458 (51 páginas)
2025-03-31 16:30:27,094 - INFO - Worker 10 procesará páginas 459 a 509 (51 páginas)
2025-03-31 16:30:27,094 - INFO - Worker 11 procesará páginas 510 a 560 (51 páginas)
2025-03-31 16:30:27,095 - INFO - Worker 12 procesará páginas 561 a 611 (51 páginas)
2025-

Iniciando scraping distribuido desde página 0 hasta 1000 con 20 workers


Scraping Mercadona:   0%|          | 0/1001 [00:00<?, ?it/s]

2025-03-31 16:30:27,107 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,109 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,110 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,110 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,111 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,112 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,112 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,114 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,115 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,115 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,117 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,117 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,119 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,119 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,120 - INFO - ====== WebDriver manager ======
2025-03-31 16:30:27,121 -

Scraping completado. Total productos: 4724
Tiempo total: 6859.37 segundos


TypeError: DataFrame.write_csv() got an unexpected keyword argument 'index'

In [6]:
df = df.with_columns(
    pl.arange(1, df.height + 1).alias("id")
)

# Reorganizar las columnas para que 'id' sea la primera
df = df.select(["id", "Category", "name", "subtitle", "price", "discount_price", "main_image_url", "secondary_image_url"])
df


# Save the DataFrame to CSV
df.to_pandas().to_csv("../data/raw/mercadona.csv", index=False)


In [3]:
df.to_pandas().to_csv("../data/raw/mercadona.csv", index=False)

## 1. Ejecutar Scraping en Paralelo

Esta versión es mucho más rápida que la original porque procesa múltiples páginas simultáneamente.

In [ ]:
# Llamar a la función optimizada para el scraping en paralelo
# Ajustar max_workers según los núcleos de tu CPU
start_time = time.time()

df = scrape_mercadona_parallel(
    start_page=0,             # Página inicial
    end_page=1000,            # Página final
    max_workers=5,            # Número de trabajadores paralelos (ajustar según CPU)
    get_secondary_images=True # Cambiar a False para hacer el scraping aún más rápido
)

end_time = time.time()
print(f"Tiempo total: {end_time - start_time:.2f} segundos")

Iniciando scraping desde página 0 hasta 1000 con 5 workers...


Scraping Mercadona:   0%|          | 0/1001 [00:00<?, ?it/s]

2025-03-31 16:09:44,108 - INFO - ====== WebDriver manager ======
2025-03-31 16:09:44,109 - INFO - ====== WebDriver manager ======
2025-03-31 16:09:44,110 - INFO - ====== WebDriver manager ======
2025-03-31 16:09:44,110 - INFO - ====== WebDriver manager ======
2025-03-31 16:09:44,111 - INFO - ====== WebDriver manager ======
2025-03-31 16:09:44,672 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,672 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,672 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,672 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,672 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,710 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,710 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 16:09:44,711 - INFO - Get LATEST chromedriver version for google-chrome
2025-03-31 

In [ ]:
# Create ID's
df = df.with_columns(
    pl.arange(1, df.height + 1).alias("id")
)

# Reorganizar las columnas para que 'id' sea la primera
df = df.select(["id", "Category", "name", "subtitle", "price", "discount_price", "main_image_url", "secondary_image_url"])
df

## 2. Descargar Imágenes en Paralelo

Esta versión de descarga es mucho más rápida que la original.

In [ ]:
# Descargar imágenes en paralelo
start_time = time.time()

download_images_parallel(
    df=df,                     # DataFrame con URLs de imágenes
    base_path="../img",         # Ruta base para guardar imágenes
    max_workers=20             # Número de descargas paralelas (mayor que el scraping)
)

end_time = time.time()
print(f"Tiempo total de descarga: {end_time - start_time:.2f} segundos")

In [ ]:
# Save the DataFrame to a CSV file.
df.write_csv("../data/processed/mercadona.csv", index=False)
print("Datos guardados correctamente en ../data/processed/mercadona.csv")